# Pipeline avancée 

Maintenant nous allons voir comment faire des pipelines avancée avec les trois fonctions suivantes :
- make_column_transformer
- make_column_selector
- make_union

In [24]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, Binarizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
import seaborn as sns

In [2]:
data = sns.load_dataset('titanic')
data.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


Ici on peut constater que nous avons différents type de variables, des variables continues comme l'age, des variables discrète comme le pclass, des variables contenant des strings comme sex, ... 

Si on désire donc le traiter avec différents transformer il faudra donc faire le tri dans nos données, si non nous allons avoir un problème, par exemple un StandardSclaer ne voudra que traiter des variables numériques, et se trouvera fort dépourvu lorsque la string fut venue.

Le problème est donc que lors de la création d'une pipeline avec un estimateur au bout on voudra passer notre dataset entier dans cette pipeline.

In [3]:
X = data.drop('survived', axis=1)
y = data['survived']

In [4]:
#model = make_pipeline(StandardScaler(), SGDClassifier())
#model.fit(X, y) # crashera

ValueError: could not convert string to float: 'male'

Pour éviter ce problème on va utiliser la fonction make_column_transformer :

Ici on écrira sous forme de tupple l'opération qui devra être transformée et la liste des colonnes qui doivent être passé à ce transformer.

In [9]:
from sklearn.compose import make_column_transformer

In [10]:
stdScalerTfrm = make_column_transformer((StandardScaler(), ['age', 'fare']))

In [11]:
stdScalerTfrm.fit_transform(X)

array([[-0.53037664, -0.50244517],
       [ 0.57183099,  0.78684529],
       [-0.25482473, -0.48885426],
       ...,
       [        nan, -0.17626324],
       [-0.25482473, -0.04438104],
       [ 0.15850313, -0.49237783]])

Ce que l'on va souvent faire c'est faire des listes contenant nos features catégorique et numériques :

In [12]:
num_f = ['pclass', 'age', 'fare']
cat_f = ['sex', 'deck', 'alone']

On pourra ensuite faire une série de transformations sur ces deux liste de features.

Pour ce faire on va créer une pipeline pour les transformations sur les valeurs numériques et une autre pour les valeurs catégoriques.

In [16]:
num_pl = make_pipeline(SimpleImputer(), StandardScaler())
cat_pl = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder())

In [18]:
preproc = make_column_transformer((num_pl, num_f), (cat_pl, cat_f))

In [19]:
model = make_pipeline(preproc, SGDClassifier())

model.fit(X, y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  ['pclass', 'age', 'fare']),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder())]),
                                                  

## Make Column Selector

Make column selector va nous permettre de créer une liste de colonnes en fonction de leurs type.

In [20]:
from sklearn.compose import make_column_selector

On va ensuite pouvoir utiliser dtype_include ou dtype_exclude pour inclure ou exclure les types de variables qui nous intéresse.

In [21]:
num_f = make_column_selector(dtype_include=np.number)
cat_f = make_column_selector(dtype_exclude=np.number)

Maintenant nous pouvons réutiliser le code vu précédement, et il nous sélectionnera automatiquement les colonnes correspondant aux types sélectionner dans notre dataset.

In [22]:
num_pl = make_pipeline(SimpleImputer(), StandardScaler())
cat_pl = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder())
preproc = make_column_transformer((num_pl, num_f), (cat_pl, cat_f))
model = make_pipeline(preproc, SGDClassifier())

model.fit(X, y)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer()),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  <sklearn.compose._column_transformer.make_column_selector object at 0x7fe593a45850>),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncode

# Make Union

make_union va nous permettre de créer des pipeline parallèle. 

SKlearn nous permet de créer plusieurs pipeline parallèle, qui vont ensuite rejoindre leurs résultats dans un seul et même tableau.

In [23]:
from sklearn.pipeline import make_union

In [35]:
numerical_features = X[['age', 'fare']]

pipeline = make_union(StandardScaler(), Binarizer())

numerical_features = numerical_features.dropna()

pipeline.fit_transform(numerical_features)

array([[-0.53037664, -0.51897787,  1.        ,  1.        ],
       [ 0.57183099,  0.69189675,  1.        ,  1.        ],
       [-0.25482473, -0.50621356,  1.        ,  1.        ],
       ...,
       [-0.73704057, -0.08877362,  1.        ,  1.        ],
       [-0.25482473, -0.08877362,  1.        ,  1.        ],
       [ 0.15850313, -0.50952283,  1.        ,  1.        ]])

Ici nous pouvons voir que nous avons eu en parallèle le standard scaler (Colonne 1 et 2) et le Binarizer (colonne 3 et 4).